In diesem Notebook soll eine akustische Datenübertragung realisiert werden. Dabei stellt dieses Skript eine Erweiterung des Demo Skriptes dar. In diesem Skript kann der Sender beliebig viele Sendenachrichten auf unterschiedliche Frequenzen mischen. Der Empfänger kann sich aus diesen gesendet Nachrichten gerade eine Nachricht herauspicken indem er die jeweilige Frequenz heruntermischt.
Die Nachricht darf ein beliebiger Text mit bis zu 255 Zeichen sein und darf keine Umlaute (ä,ö,ü) beinhalten. 
Mit UTF-8 wird jedes Zeichen der Nachricht in eine 8-Bit Nachricht encodiert.
Um später im Empfänger den Beginn der Nachricht zu detektieren wird eine Präambel sowie die Länge der Nachricht vor die eigentliche Nachricht gestellt.
Die so erhaltene Bitfolge wird mit einer BPSK-Modulation moduliert, mit einer senderseitigen Pulsformung (Recht eck oder Root Raised Cosine) gefaltet und mit einer harmonischen Schwingung mit Trägerfrequenz in den Bandpassbereich gemischt.

Das erhaltene Audiosignal kann über Lautsprecher abgespielt und mit einem Mikrofon aufgenommen werden (Empfangssignal).

Das Empfangssignal wird runtergemischt und anschließend mit dem Matched-Filter gefaltet. 
Die empfängerseitig bekannte Präambel wird im Empfangssignal gesucht, die Länge der Nachricht bestimmt und die entsprechende Sequenz aus dem Empfangssignal extrahiert. 
Die BPSK-Modulation wird invertiert und mit UTF-8 wird die eigentliche Nachricht rekonstruiert. 

In [ ]:
# Import all python dependencies that we need!
import numpy as np
#%matplotlib notebook
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import scipy
from scipy.io.wavfile import write, read

In [ ]:
# Define you number of carriers
num_carrier = 3

# Define your message to transmit
message_tx = [None] * num_carrier
message_tx[0] = 'Mamaaaaaa'
message_tx[1] = 'Papa'
message_tx[2] = 'Kind'

for carrier_idx in range(num_carrier):
    for x in enumerate(list(message_tx[carrier_idx])):
        if x[1] == ('ä') or x[1] == ('ö') or x[1] == ('ü'):
            print('Fehler: Nachricht darf keine Umlaute beinhalten!')      

    if np.size(list(message_tx[carrier_idx]))>2**8-1:
        print('Fehler: Die Länge der Nachricht überschreitet die maximale Gesamtlänge!')


Nun werden die Parameter der Übertragung definiert.
Die Symboldauer $T$, die Trägerfrequenz $f_\mathrm{T}$, die Abtastrate des Audiosignals (und damit des Bandpasssignals).
Der Überabtastfaktor pro Symbol berechnet sich aus der Abtastrate und der Symboldauer.
Als Präambel kann entweder ein Barker-Code oder eine Gold-Folge gewählt werden, da beides binäre Folgen mit hervorragenden Autokorrelationseigenschaften sind.
Weiterhin muss das Pulsformungsfilter gewählt werden, entweder ein Rechteckfilter (rect) oder ein Root-Raised-Cosine-Filter (rrc). Für letzteres kann noch der Roll-off-Faktor $\beta$ und die Länge der Impulsantwort in Vielfachen der Symboldauer gewählt werden.
Zum Schutz vor Übertragungsstörungen werden für $n_\mathrm{header}~>~1,\, n_\mathrm{header} \in \mathbb{N},$ wichtige Daten (Header) mit einem $n_\mathrm{header}$-Wiederholungscode kodiert. Die Informationsdaten können mit $n_\mathrm{data} > 1,\, n_\mathrm{data} \in \mathbb{N},$ durch einen Wiederholungscode geschützt werden.

In [ ]:
#Define variables for transmission
T = 0.1                         # symbol duration in s
f_t = [261.63, 329.63, 392.00]  # carrier frequency
rate = np.int32(44100)          # rate at which audio signal is sampled
n_up = np.int32(rate*T)         # oversampling factor -> samples per symbol
preamble_type = 'gold'          # 'gold' or 'barker'
filt_type = 'rrc'               # Either 'rect' ir 'rrc'
beta= 0                         # 0 <= beta <= 1;  Roll-off factor of rrc, 0 -> sinc(t)
K = 8                           # Length of rrc in symbols, must be even number
n_header = 5                    # Blocklength of repetition code for header: How often bit is repeated
n_data = 1                      # Blocklength of rep. code for information data
path = 'audio/'                 # where to save audio file

In [ ]:
# Define functions to convert text-strings into arrays of bits and vice versa
def text2bits(message): 
    message= list(message)
    bits = []
    for d1 in range(np.size(message)):
        dummy = f'{ord(message[d1]):08b}'       #Convert char to integer number and then into binary representation
        bits.append(dummy)                  
    bit_out = ''.join(str(x) for x in bits)
    return np.int32(list(bit_out))              #return numpy array with bits

In [ ]:
def rep_encode(bits,n):
    '''
    Construct an (n,1)-repetition code given the data bits
    IN: binary data to encode, blocklength of repetition code
    OUT: encoded binary sequence
    '''
    c = np.repeat(np.expand_dims(bits,axis=1),n,axis=1)     #repeat every bit n times in matrix
    return np.matrix.flatten(c)      #reshape matrix to 1D-array

Als Pulsformungsfilter wird ein Rechteckfilter oder ein Root-Raised-Cosine Filter genutzt. Wir definieren beide sowie eine Funktion, welche beide entsprechend der Auswahl berechnet sowie ihre Leistung normiert:

In [ ]:
# Get rectangle-function for pulse shaping and normalize energy of pulse to 1 
def get_rect(n_up):
    '''
    Determines the coefficients of a rectangular filter
    IN: length of IR
    OUT: filter coefficients
    '''
    rect = np.ones(n_up)
    rect/=np.linalg.norm(rect)
    return rect

def get_rrc(K, n_up, T, beta):
    
    ''' 
    Determines coefficients of an RC filter 
    
    Formula out of: John B. Anderson: Digital transmission engineering.
    
    NOTE: Length of the IR has to be an odd number
    
    IN: length of IR in symbols, upsampling factor per symbol, symbol time, roll-off factor
    OUT: filter coefficients
    '''
    N = K*n_up+1                        #number of samples for filter
    T_delta = T/n_up                    #time resolution of upsampled filter
    sample_num = np.arange(N)    
    h_rrc = np.zeros(N, dtype=float)

    for x in sample_num:
        t = np.array([(x-N/2)*T_delta])
        if t == 0.0:
            scaling = 1/np.sqrt(T)
            equation = 1-beta+(4*beta/np.pi)
            h_rrc[x] = scaling * equation
        elif beta != 0 and t == (T/(4*beta) or -T/(4*beta)):
            scaling = beta/np.sqrt(2*T)
            equation = (1+(2/np.pi))*np.sin(np.pi/(4*beta)) + (1-2/np.pi)*np.cos(np.pi/(4*beta))
            h_rrc[x] = scaling * equation
        else:
            scaling = 1/np.sqrt(T)
            numerator = np.sin(np.pi*(1-beta)*t/T) + (4*beta*t/T)*np.cos(np.pi*(1+beta)*t/T)
            denominator = (np.pi*t/T)*(1-np.square(4*beta*t/T))
            equation = numerator / denominator
            h_rrc[x] = scaling * equation
            if denominator == 0:            #Dirty workarround for denom=0, negligible if n_up is sufficiently big
                h_rrc[x] = h_rrc[x-1]           
    return h_rrc 

def get_filt(filt_type, n_up, K, T, beta):
    ''' 
    Return the impulse response of an rrc or a rect filter with energy normalized to one
    IN: rrc or rect, upsampling factor per symbol, length of rrc in symbols, symbol duration, beta of rrc 
    OUT: filter coefficients
    '''
    if filt_type =='rrc':
        filter = get_rrc(K,n_up,T, beta)
    elif filt_type =='rect':
        filter = get_rect(n_up)     #for rect K is always 1
    else: 
        print('No valid filter type! Either rect or rrc, but '+filt_type+' was passed to function.')
    filter /=np.linalg.norm(filter)
    return filter


Vor der Übertragung wird die Präambel definiert. Diese ist dem Empfänger bekannt und dient am Empfänger unter anderem der Detektion des Beginns des Nutzsignals.
Zur Auswahl stehen ein Barker-Code oder eine Gold-Folge, wobei beide hervorragende Autokorrelationseigenschaften besitzen.

In [ ]:
# Choose a predefined preamble (known at the receiver)
# Gold-sequence taken from https://ieeexplore.ieee.org/stamp/stamp.jsp?tp=&arnumber=922961
gold = np.array([0,1,0,1,1,0,0,1,0,1,1,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,1,1,0,1,0])  # Gold sequence
barker = np.array([0,0,0,0,0,1,1,0,0,1,0,1,0])              #Barker-13-Code

if preamble_type == 'gold':
    preamble = gold
elif preamble_type == 'barker':
    preamble = barker 
else: 
    print('Keine gültige Folge als Präambel ausgewählt!')

Die zu sendende Nachricht (Nutzdaten) wird in einen Array aus Bits umgewandelt.
Danach wird die Länge der Nachricht bestimmt, diese wird binär dargestellt.
Die komplette Nachricht besteht aus Präambel, Informationen über die Länge der Nachricht sowie den Nutzdaten.
Abschließend wird jedes Bit $b$ durch $x=(-1)^b$ in ein BPSK-Symbol $x \in \{-1,+1\}$ umgewandelt.

In [ ]:
data = [None] * num_carrier
data_length = [None] * num_carrier
num_chars = [None] * num_carrier
bit = [None] * num_carrier
sym = [None] * num_carrier

for carrier_idx in range(num_carrier):
    #Convert text into bits
    data[carrier_idx] = np.int32(list(text2bits(message_tx[carrier_idx])))                #Convert message into array of bits
    data[carrier_idx] = rep_encode(data[carrier_idx],n_data)

    #Count number of characters that are transmitted, convert into binary representation and apply n-repetition code 
    data_length[carrier_idx]=np.zeros(8)                                     #Empty array to parse message length into
    num_chars[carrier_idx] = list(np.binary_repr(np.size(list(message_tx[carrier_idx]))))    #Count number of characters in message
    data_length[carrier_idx][-np.size(num_chars[carrier_idx]):] = np.int32(num_chars[carrier_idx])     #convert message length into binary representation
    data_length[carrier_idx] = rep_encode(data_length[carrier_idx],n_header)                     #apply channel coding to protect these bits  
    
    # Concatenate preamble, data length and data, then do BPSK modulation
    bit[carrier_idx] = np.concatenate((preamble,data_length[carrier_idx],data[carrier_idx]))       #Add preamble for start of transmission and data length
    sym[carrier_idx] = (-1)**bit[carrier_idx]                                         #Apply BPSK-modulation

Das Pulsformungsfilter $g(t)$ wird definiert.
Um die Überabtastung zu realisieren werden in einem Array mit Nullen nach jeder $n_\mathrm{up}$-ten Stelle die zu sendenden BPSK-Symbole platziert.
Durch $s(t)=\sum_k x[k] g(t-kT)$ wird das Basisbandsignal erzeugt, im Code durch eine Faltung realisiert:

In [ ]:
sym_up = [None] * num_carrier
tx_bb = [None] * num_carrier

d1 = 5                                  #number of symbols for zoom of plot
fig, axes = plt.subplots(nrows=3, ncols=num_carrier, sharex='col', figsize=(12, 8))
off = (np.size(preamble)+n_header*8)*n_up-int(n_up/10)      #Offset to plot info data, not preamble or data length info

for carrier_idx in range(num_carrier):
    #Upsample symbols and convolve with pulse shaping filter
    filter = get_filt(filt_type, n_up, K, T, beta)      #Get pulse shaping filter
    sym_up[carrier_idx] = np.zeros(np.size(bit[carrier_idx])*n_up)                #Create new array for zero-padded signal     
    sym_up[carrier_idx][::n_up]=sym[carrier_idx]                                  #Paste symbols into new array        
    tx_bb[carrier_idx] = scipy.signal.fftconvolve(filter,sym_up[carrier_idx])     #Create BB-signal by convolution with pulse shaping filter

    #Plot zoomed version of baseband signal for first d1 netto symbols
    t_plot = np.arange(0,np.size(tx_bb[carrier_idx])-1,1)/rate
    
    ax1 = axes[0, carrier_idx] if num_carrier > 1 else axes[0]
    ax1.stem(t_plot[off:off+d1*n_up],np.zeros((d1*n_up)))
    ax1.grid(True)
    ax1.set_title('Leeres Array')
    ax1.set_ylim(-1.2,1.2)

    ax2 = axes[1, carrier_idx] if num_carrier > 1 else axes[1]
    ax2.stem(t_plot[off:off+d1*n_up],sym_up[carrier_idx][off:off+d1*n_up])
    ax2.set_title('BPSK-Symbole')
    ax2.set_ylim(-1.2,1.2)
    ax2.set_ylabel('$\sum_k x[k] \\delta(t-kT)$')
    ax2.grid(True)

    ax3 = axes[2, carrier_idx] if num_carrier > 1 else axes[2]
    ax3.plot(t_plot[off:off+d1*n_up],tx_bb[carrier_idx][off+np.int32(K/2)*n_up:off+(d1+np.int32(K/2))*n_up])
    ax3.set_title('Basisbandsignal')
    ax3.set_xlim(off/n_up*T,((off+2*n_up/10)/n_up+(d1-1))*T)
    ax3.set_xlabel('time [s]')
    ax3.set_ylabel('$s(t)$')
    ax3.grid(True)

    fig.tight_layout()
    fig.show()

Durch $s_\mathrm{BP}(t)=\mathrm{Re} \left\{ s(t) \mathrm{e}^{\mathrm{j}2\pi f_\mathrm{T} t}\right\}$ wird aus dem Basisbandsignal $s(t)$ das Bandpasssignal $s_\mathrm{BP}(t)$ erzeugt:

In [ ]:
t = [None] * num_carrier
carrier = [None] * num_carrier
tx = [None] * num_carrier

for carrier_idx in range(num_carrier):
    #Convert BB -> BP
    t[carrier_idx] = np.linspace(0,np.size(tx_bb[carrier_idx])-1,np.size(tx_bb[carrier_idx]))/rate
    carrier[carrier_idx] = np.exp(1j*2*np.pi*f_t[carrier_idx]*t[carrier_idx])
    tx[carrier_idx] = np.real(tx_bb[carrier_idx] * carrier[carrier_idx]) 

Nun können wir die einzelnen Unterträger additiv überlagern

In [ ]:
max_len = max(len(sublist) for sublist in tx)

# Add zeros after when the transmission of the previous data streams has ended
for carrier_idx in range(num_carrier):
    tx[carrier_idx] = np.pad(tx[carrier_idx], (0, max_len - len(tx[carrier_idx])), mode='constant')
    
t_sum = np.linspace(0,max_len-1,max_len)/rate
tx_sum = sum(tx)

Wir können nun das Basisbandsignal und das Bandpasssignal sowie deren Spektren plotten: 

In [ ]:
d2 = 5

fig, axes = plt.subplots(nrows=2, ncols=num_carrier, figsize=(12, 8))
cmap_tx = cm.get_cmap('Pastel1', num_carrier)  # use tab10 with num_carrier distinct colors
cmap_tx_bb = cm.get_cmap('Set1', num_carrier)  # use tab10 with num_carrier distinct colors
off = (np.size(preamble)+n_header*8)*n_up      #Offset to plot info data, not preamble or data length info

for carrier_idx in range(num_carrier):
    # Plot baseband and bandpass of first d2 netto data symbols
    ax1 = axes[0, carrier_idx] if num_carrier > 1 else axes[0]
    ax1.plot(t[carrier_idx][off:off+d2*n_up],tx[carrier_idx][off+np.int32(K/2)*n_up:off+(d1+np.int32(K/2))*n_up],label='BP',color=cmap_tx(carrier_idx),linewidth=0.1)
    ax1.plot(t[carrier_idx][off:off+d2*n_up],tx_bb[carrier_idx][off+np.int32(K/2)*n_up:off+(d1+np.int32(K/2))*n_up],label='BB',color=cmap_tx_bb(carrier_idx))
    ax1.grid(True)
    ax1.legend(loc='center left')
    ax1.set_xlim(off/n_up*T,(off/n_up+d2)*T)
    ax1.set_xlabel('time [s]')
    ax1.set_xticks(ticks=(np.arange(off/n_up,off/n_up+d2,1)*T),labels=(np.round(np.arange(off/n_up,off/n_up+d2,1)*T,1)))
    ax1.set_ylabel('$s(t)$, $s_\\mathrm{BP}(t)$')
    ax1.set_title('Signale')

    #Calculate spectra and normalize them to max. amplitude = 1, then plot everyhting
    S = np.abs(np.fft.fftshift(np.fft.fft(tx_bb[carrier_idx])))
    S_BP = np.abs(np.fft.fftshift(np.fft.fft(tx[carrier_idx])))

    ax2 = axes[1, carrier_idx] if num_carrier > 1 else axes[1]
    ax2.plot(np.linspace(-rate/2,rate/2,np.size(tx[carrier_idx]))/rate,S_BP,color=cmap_tx(carrier_idx), label='BP')
    ax2.plot(np.linspace(-rate/2,rate/2,np.size(tx_bb[carrier_idx]))/rate,S,color=cmap_tx_bb(carrier_idx),label='BB')
    ax2.grid(True)
    ax2.set_xlim(-2*(f_t[0]/rate),2*f_t[0]/rate)
    ax2.set_xticks(ticks=(np.array([-2,-1,0,1,2])*(f_t[0]/rate)),labels=(-2,-1,0,1,2))
    ax2.set_xlabel('$f/f_\\mathrm{T0}$')
    ax2.set_ylabel('|$S(f)$|, |$S_\\mathrm{BP}(f)$|')
    ax2.legend(loc='center left')
    ax2.set_title('Spektren')
    
fig.tight_layout()
fig.show()

In [ ]:
#Calculate spectra and normalize them to max. amplitude = 1, then plot everyhting
S_BP_sum = np.abs(np.fft.fftshift(np.fft.fft(tx_sum)))

fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(12, 8))
ax1.plot(t_sum[off:off+d2*n_up],tx_sum[off+np.int32(K/2)*n_up:off+(d1+np.int32(K/2))*n_up],linewidth=0.1)
ax1.grid(True)
ax1.set_xlim(off/n_up*T,(off/n_up+d2)*T)
ax1.set_xlabel('time [s]')
ax1.set_xticks(ticks=(np.arange(off/n_up,off/n_up+d2,1)*T),labels=(np.round(np.arange(off/n_up,off/n_up+d2,1)*T,1)))
ax1.set_ylabel('$s_\\mathrm{BP}(t)$')
ax1.set_title('Zeitbereich')

ax2.plot(np.linspace(-rate/2,rate/2,np.size(tx_sum))/rate,S_BP_sum)
ax2.grid(True)
ax2.set_xlim(-2*(f_t[0]/rate),2*f_t[0]/rate)
ax2.set_xticks(ticks=(np.array([-2,-1,0,1,2])*(f_t[0]/rate)),labels=(-2,-1,0,1,2))
ax2.set_xlabel('$f/f_\\mathrm{T0}$')
ax2.set_ylabel('|$S_\\mathrm{BP}(f)$|')
ax2.legend(loc='center left')
ax2.set_title('Spektren')

Wir speichern die Audio-Datei und können diese nun abspielen:

In [ ]:
# Save audio signal and preamble
scaled = np.int16(tx_sum / np.max(np.abs(tx_sum)) * 32767)
write(path+'tx.wav', np.int32(rate), scaled)

# Now we can play our audio signal!
import IPython 
IPython.display.Audio(path+'tx.wav')

Hier kann das Audio-Signal nun übertragen werden, sprich von einem Lautsprecher abgespielt werden.